# 🚦 Smart Toll Pricing System
## Phase 2 — Machine Learning Model Training

| | |
|---|---|
| **Student** | Nitin Rajgor |
| **University** | Jain (Deemed-to-be) University, Bengaluru |
| **Paper** | IRE Journals, Vol. 9, Issue 11, May 2026 |

---
### What we do here:
- Load processed data from Phase 1
- Engineer 17 powerful features
- Train Random Forest Classifier
- Evaluate: Accuracy, Precision, Recall, F1
- Visualize: Confusion Matrix + Feature Importance
- Save trained model

## 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
import joblib, json, os
os.makedirs('models', exist_ok=True)

print('✅ All libraries imported!')

---
## 📁 Step 1 — Load Processed Data

In [ ]:
df = pd.read_csv('data/processed_traffic.csv')

print('='*50)
print('  PROCESSED DATA LOADED')
print('='*50)
print(f'  Total Rows    : {len(df):,}')
print(f'  Total Columns : {df.shape[1]}')
print()
print('  Congestion Distribution:')
dist = df['congestion_level'].value_counts().sort_index()
for k,v in dist.items():
    print(f'    {["Low","Medium","High"][k]:8s}: {v:6,} ({v/len(df)*100:.1f}%)')

df.head()

---
## ⚙️ Step 2 — Feature Engineering
Creating 17 smart features for the ML model

In [ ]:
# Feature 1: Rush hour intensity
def rush_intensity(h):
    if 7<=h<=9 or 16<=h<=18: return 2   # Peak
    elif 10<=h<=11 or 14<=h<=15: return 1 # Near-peak
    return 0                              # Off-peak

# Feature 2: Time of day
def time_of_day(h):
    if 5<=h<12: return 0    # Morning
    elif 12<=h<17: return 1  # Afternoon
    elif 17<=h<21: return 2  # Evening
    return 3                 # Night

# Feature 3: Season
def get_season(m):
    if m in [12,1,2]: return 0   # Winter
    elif m in [3,4,5]: return 1  # Spring
    elif m in [6,7,8]: return 2  # Summer
    return 3                     # Autumn

df['rush_intensity']  = df['hour'].apply(rush_intensity)
df['time_of_day']     = df['hour'].apply(time_of_day)
df['season']          = df['month'].apply(get_season)
df['vol_speed_ratio'] = (df['traffic_volume'] / (df['avg_speed']+1)).round(3)
df['bad_weather']     = ((df['weather_encoded']>=2)|(df['rain_1h']>0)|(df['snow_1h']>0)).astype(int)
df['day_type']        = df.apply(lambda r: 2 if r['is_holiday']==1 else (1 if r['is_weekend']==1 else 0), axis=1)

print('Features Created:')
print('  rush_intensity  → 0=off-peak, 1=near-peak, 2=peak')
print('  time_of_day     → 0=Morning, 1=Afternoon, 2=Evening, 3=Night')
print('  season          → 0=Winter, 1=Spring, 2=Summer, 3=Autumn')
print('  vol_speed_ratio → traffic_volume / avg_speed (congestion proxy)')
print('  bad_weather     → 1=rain/snow/fog, 0=clear')
print('  day_type        → 0=Weekday, 1=Weekend, 2=Holiday')

FEATURE_COLS = [
    'hour','day_of_week','month',
    'rush_intensity','time_of_day','season','day_type',
    'traffic_volume','avg_speed','travel_time','vol_speed_ratio',
    'temp_celsius','rain_1h','snow_1h','clouds_all',
    'weather_encoded','bad_weather'
]

df[FEATURE_COLS + ['congestion_level']].to_csv('data/final_features.csv', index=False)
print(f'\n✅ Total Features: {len(FEATURE_COLS)}')
print('✅ Saved → data/final_features.csv')

---
## ✂️ Step 3 — Train / Test Split
80% Training | 20% Testing

In [ ]:
X = df[FEATURE_COLS]
y = df['congestion_level']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('='*50)
print('  DATA SPLIT')
print('='*50)
print(f'  Total Samples  : {len(X):,}')
print(f'  Training Set   : {len(X_train):,} (80%)')
print(f'  Testing Set    : {len(X_test):,} (20%)')
print(f'  Features Used  : {len(FEATURE_COLS)}')
print()
print('✅ Split complete!')

---
## 🤖 Step 4 — Train Random Forest Model
**Why Random Forest?** (Paper Section 3.2.4)
- Works well on large datasets
- No need for data scaling
- Handles non-linear traffic patterns
- Gives feature importance scores

In [ ]:
print('Training Random Forest Classifier...')
print('100 decision trees being built...')
print()

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

print('✅ Model trained successfully!')
print(f'   Trees in forest : {model.n_estimators}')
print(f'   Features used   : {model.n_features_in_}')

---
## 📊 Step 5 — Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec  = recall_score(y_test, y_pred, average='weighted')
f1   = f1_score(y_test, y_pred, average='weighted')

print('='*50)
print('  MODEL PERFORMANCE')
print('='*50)
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  Precision : {prec*100:.2f}%')
print(f'  Recall    : {rec*100:.2f}%')
print(f'  F1 Score  : {f1*100:.2f}%')
print()
print('--- Per Class Report ---')
print(classification_report(y_test, y_pred, target_names=['Low','Medium','High']))

In [ ]:
# 5-Fold Cross Validation
cv = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print('--- 5-Fold Cross Validation ---')
for i,s in enumerate(cv,1):
    bar = '█'*int(s*50)
    print(f'  Fold {i}: {s*100:.2f}%  {bar}')
print(f'\n  Mean Accuracy : {cv.mean()*100:.2f}%')
print(f'  Std Deviation : {cv.std()*100:.2f}%')
print()
if cv.mean() >= 0.80:
    print('✅ Model is GOOD — above 80% target!')
else:
    print('⚠️ Model needs improvement')

---
## 📈 Step 6 — Visualizations

In [ ]:
# Chart 1: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7,5))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i][j]), ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if cm[i][j]>cm.max()/2 else 'black')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(['Low','Medium','High'])
ax.set_yticklabels(['Low','Medium','High'])
ax.set_xlabel('Predicted', fontsize=12); ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix — Diagonal = Correct Predictions', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('static/charts/confusion_matrix.png', dpi=120); plt.show()
print('💡 Diagonal values = correct predictions. Higher = better!')

In [ ]:
# Chart 2: Feature Importance
fi = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10,7))
colors = ['#dc3545' if v>0.1 else '#1a237e' if v>0.05 else '#90a4ae' for v in fi.values]
ax.barh(fi.index, fi.values, color=colors, height=0.6)
ax.set_xlabel('Importance Score')
ax.set_title('Feature Importance — Which Features Matter Most?', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.legend(handles=[
    mpatches.Patch(color='#dc3545', label='>10% Very Important'),
    mpatches.Patch(color='#1a237e', label='5-10% Important'),
    mpatches.Patch(color='#90a4ae', label='<5% Less Important')
], loc='lower right')
plt.tight_layout(); plt.savefig('static/charts/feature_importance.png', dpi=120); plt.show()
print('💡 Top features = most important for predicting congestion!')

In [ ]:
# Chart 3: Metrics Bar Chart
metrics = {'Accuracy':acc,'Precision':prec,'Recall':rec,'F1 Score':f1}
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(metrics.keys(), [v*100 for v in metrics.values()],
              color=['#1a237e','#28a745','#ffc107','#dc3545'], width=0.5)
ax.set_ylabel('Score (%)'); ax.set_ylim(0,110)
ax.set_title('Model Performance Metrics', fontsize=13, fontweight='bold')
ax.axhline(y=80, color='red', linestyle='--', alpha=0.5, label='80% target')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for bar,val in zip(bars,metrics.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{val*100:.1f}%', ha='center', fontweight='bold', fontsize=11)
plt.tight_layout(); plt.savefig('static/charts/model_metrics.png', dpi=120); plt.show()

---
## 💾 Step 7 — Save Model

In [ ]:
joblib.dump(model, 'models/rf_model.pkl')

report  = classification_report(y_test, y_pred, output_dict=True)
fi_dict = {col: float(round(val,4)) for col,val in zip(FEATURE_COLS, model.feature_importances_)}

metrics_data = {
    'accuracy':  round(float(acc),4),
    'precision': round(float(prec),4),
    'recall':    round(float(rec),4),
    'f1_score':  round(float(f1),4),
    'cv_mean':   round(float(cv.mean()),4),
    'report':    report,
    'feature_importance': fi_dict
}
with open('models/metrics.json','w') as f:
    json.dump(metrics_data, f, indent=2)

print('✅ Model saved   → models/rf_model.pkl')
print('✅ Metrics saved → models/metrics.json')
print()
print('These files are used by the Flask Web App!')

---
## ✅ Summary

In [ ]:
print('='*55)
print('  MODEL TRAINING — COMPLETE')
print('='*55)
print(f'  Model      : Random Forest Classifier')
print(f'  Trees      : 100')
print(f'  Features   : {len(FEATURE_COLS)}')
print(f'  Train rows : {len(X_train):,}')
print(f'  Test rows  : {len(X_test):,}')
print()
print(f'  Accuracy   : {acc*100:.2f}%')
print(f'  F1 Score   : {f1*100:.2f}%')
print(f'  CV Mean    : {cv.mean()*100:.2f}%')
print()
print('  Saved:')
print('  → models/rf_model.pkl')
print('  → models/metrics.json')
print()
print('  Next Step → Open: Pricing_Engine.ipynb')
print('='*55)